In [23]:
#Extract text from PDF (PyMuPDF / fitz)
import fitz  # fitz is the module name for PyMuPDF library

def extract_text_from_pdf(pdf_path):
    # Open the PDF file from the given path
    # fitz.open() loads the entire PDF into memory as a document object
    doc = fitz.open(pdf_path)
    
    # This will hold all the text from every page combined
    full_text = ""
    
    # Loop through each page — enumerate gives us both index and page object
    # page_num starts at 0, so we add 1 for human-readable display
    for page_num, page in enumerate(doc):
        
        # Extract raw plain text from this page
        # get_text() strips all PDF formatting and returns just the words
        text = page.get_text()
        
        # Print a preview of this page (first 200 chars only)
        # Just so we can see whats happening as the code runs
        print(f"--- Page {page_num + 1} ---")
        print(text[:200])
        
        # Append this pages text to our full document text
        full_text += text
    
    # Always close the document after reading to free memory
    doc.close()
    
    # Return the entire text as one big string
    # Next step we will break this into smaller chunks
    return full_text

In [7]:
# Pass our sample PDF path to the function
# Make sure sample.pdf is in the same folder as your notebook
text = extract_text_from_pdf("sample.pdf")
print(f"text - {text}")
print("\n--- DONE ---")

# len() tells us total number of characters extracted
# This gives us a sense of how large the document is
print(f"Total characters extracted: {len(text)}")

--- Page 1 ---
Group Insurance Plan — Member Policy Guide
1. Policy Overview
This document outlines the terms and conditions of your group insurance policy. Your policy provides
comprehensive coverage including heal
--- Page 2 ---
Claims Submission Process
4. How to Submit a Claim
All claims must be submitted within 90 days of the date of service. Claims submitted after this deadline
may be denied unless exceptional circumstanc
--- Page 3 ---
Health and Dental Coverage Details
8. Dental Benefits
Basic dental services including examinations, X-rays, cleanings, and fillings are covered at 80 percent
of the fee guide. Major restorative servic
--- Page 4 ---
Disability Benefits and Appeals Process
12. Short-Term Disability
Short-term disability (STD) benefits replace 66.67 percent of your weekly earnings if you become totally
disabled due to illness or in
text - Group Insurance Plan — Member Policy Guide
1. Policy Overview
This document outlines the terms and conditions of your group insur

In [8]:
def chunk_text(text, chunk_size=500, chunk_overlap=100):
    chunks = []
    start = 0

    while start < len(text):
        # Take a slice of text from start to start+chunk_size
        end = start + chunk_size
        chunk = text[start:end]
        chunks.append(chunk)

        # Move forward by chunk_size minus overlap
        # This ensures consecutive chunks share some context
        start += chunk_size - chunk_overlap

    return chunks

In [9]:
# Clean minimal whitespace first
clean_text = " ".join(text.split())

# Chunk it
chunks = chunk_text(clean_text)

print(f"Total chunks created: {len(chunks)}")
print(f"\n--- Chunk 1 ---\n{chunks[0]}")
print(f"\n--- Chunk 2 ---\n{chunks[1]}")
print(f"\n--- Chunk 3 ---\n{chunks[2]}")

Total chunks created: 14

--- Chunk 1 ---
Group Insurance Plan — Member Policy Guide 1. Policy Overview This document outlines the terms and conditions of your group insurance policy. Your policy provides comprehensive coverage including health, dental, vision, and disability benefits. All plan members are entitled to submit claims as per the guidelines described in this document. 2. Eligibility To be eligible for coverage, plan members must be actively employed on a full-time basis for a minimum of 20 hours per week. Coverage begins on

--- Chunk 2 ---
ust be actively employed on a full-time basis for a minimum of 20 hours per week. Coverage begins on the first day of the month following your hire date. Dependents including spouse and children under age 21 may be added to your plan within 31 days of a qualifying life event such as marriage or birth of a child. 3. Effective Date of Coverage Coverage becomes effective on the date specified in your enrollment confirmation letter. If you 

In [10]:
import ollama

def get_embedding(text):
    # Send text to nomic-embed-text model running in Ollama
    # It returns a vector — a list of float numbers
    # Similar texts will have similar vectors
    response = ollama.embeddings(
        model="nomic-embed-text",
        prompt=text
    )
    
    # Extract just the embedding list from the response
    return response["embedding"]


In [11]:
# Test it on a single chunk first before embedding all 14
sample_embedding = get_embedding(chunks[0])

print(f"Embedding type: {type(sample_embedding)}")
print(f"Embedding length: {len(sample_embedding)}")
print(f"First 5 values: {sample_embedding[:5]}")


Embedding type: <class 'list'>
Embedding length: 768
First 5 values: [0.005495536141097546, 0.8076394200325012, -3.43955135345459, -0.5675997138023376, -0.19819042086601257]


In [12]:
def build_vector_store(chunks):
    vector_store = []
    
    for i, chunk in enumerate(chunks):
        print(f"Embedding chunk {i + 1}/{len(chunks)}...")
        
        # Get the embedding vector for this chunk
        embedding = get_embedding(chunk)
        
        # Store both the original text AND its embedding together
        # We need the text later to send to the LLM
        # We need the embedding now to do similarity search
        vector_store.append({
            "text": chunk,
            "embedding": embedding
        })
    
    return vector_store


In [13]:
# This will take a few seconds — embedding 14 chunks one by one
vector_store = build_vector_store(chunks)

print(f"\nVector store built with {len(vector_store)} entries")
print(f"Each entry has keys: {vector_store[0].keys()}")


Embedding chunk 1/14...
Embedding chunk 2/14...
Embedding chunk 3/14...
Embedding chunk 4/14...
Embedding chunk 5/14...
Embedding chunk 6/14...
Embedding chunk 7/14...
Embedding chunk 8/14...
Embedding chunk 9/14...
Embedding chunk 10/14...
Embedding chunk 11/14...
Embedding chunk 12/14...
Embedding chunk 13/14...
Embedding chunk 14/14...

Vector store built with 14 entries
Each entry has keys: dict_keys(['text', 'embedding'])


In [14]:
import math

def cosine_similarity(vec_a, vec_b):
    # Dot product: multiply matching elements and sum them
    # This tells us how much the vectors "agree" in direction
    dot_product = sum(a * b for a, b in zip(vec_a, vec_b))
    
    # Magnitude of vector A: sqrt of sum of squares
    magnitude_a = math.sqrt(sum(a ** 2 for a in vec_a))
    
    # Magnitude of vector B: same idea
    magnitude_b = math.sqrt(sum(b ** 2 for b in vec_b))
    
    # Final cosine similarity score between -1 and 1
    # 1 = identical direction, 0 = unrelated, -1 = opposite
    return dot_product / (magnitude_a * magnitude_b)


In [15]:
def retrieve(query, vector_store, top_k=3):
    # Step 1: Convert the user's question into an embedding
    # using the SAME embedding model we used for chunks
    query_embedding = get_embedding(query)
    
    # Step 2: Compare this query vector against EVERY chunk vector
    scored_chunks = []
    for item in vector_store:
        score = cosine_similarity(query_embedding, item["embedding"])
        scored_chunks.append((score, item["text"]))
    
    # Step 3: Sort by score, highest similarity first
    scored_chunks.sort(key=lambda x: x[0], reverse=True)
    
    # Step 4: Return only the top_k most relevant chunks
    return scored_chunks[:top_k]


In [16]:
query = "What is the deadline to submit a claim?"

results = retrieve(query, vector_store, top_k=3)

for score, text in results:
    print(f"Score: {score:.4f}")
    print(f"Text: {text}\n")


Score: 0.8105
Text:  enrollment confirmation letter. If you miss your enrollment window, you may be required to provide evidence of insurability before coverage is granted. Late applicants will have coverage deferred until the next open enrollment period. Claims Submission Process 4. How to Submit a Claim All claims must be submitted within 90 days of the date of service. Claims submitted after this deadline may be denied unless exceptional circumstances prevented timely submission. Members can submit claims online

Score: 0.7135
Text: ified by education, training, or experience. 14. Claim Denials and Appeals If your claim is denied, you will receive a written explanation detailing the reason for denial. You have the right to appeal the decision within 90 days of receiving the denial notice. To initiate an appeal, submit a written request along with any additional supporting documentation to the Appeals Department. A senior claims adjudicator will review your appeal and respond within 

In [17]:
def build_prompt(query, retrieved_chunks):
    # Combine all retrieved chunk texts into one context block
    context = "\n\n".join([text for score, text in retrieved_chunks])
    
    # This prompt template is crucial — it tells the LLM:
    # 1. Only use the provided context
    # 2. Don't make things up if the answer isn't there
    prompt = f"""You are a helpful assistant answering questions based ONLY on the context below.
If the answer is not in the context, say "I don't have enough information to answer that."

Context:
{context}

Question: {query}

Answer:"""
    
    return prompt


In [18]:
def ask(query, vector_store, top_k=3):
    # Step 1: Retrieve relevant chunks
    retrieved_chunks = retrieve(query, vector_store, top_k=top_k)
    
    # Step 2: Build the prompt with context
    prompt = build_prompt(query, retrieved_chunks)
    
    # Step 3: Send to Gemma via Ollama
    response = ollama.chat(
        model="gemma3",
        messages=[{"role": "user", "content": prompt}]
    )
    
    return response["message"]["content"]


In [20]:
answer = ask("What is the deadline to submit a claim?", vector_store)
print(answer)


Claims must be submitted within 90 days of the date of service.


In [21]:
print(ask("How much is dental coverage for major services?", vector_store))
print(ask("What happens if my claim is denied?", vector_store))


Major restorative services such as crowns, bridges, and dentures are covered at 50 percent.
If your claim is denied, you will receive a written explanation detailing the reason for denial. You have the right to appeal the decision within 90 days of receiving the denial notice. To initiate an appeal, submit a written request along with any additional supporting documentation to the Appeals Department.


In [22]:
print(ask("what is AI",vector_store))

I don't have enough information to answer that.
